# Video Translation Pipeline - Debug Notebook

This notebook provides an interactive debugging environment for the cloud video translation pipeline.
Each stage of the pipeline is broken into separate cells, allowing you to:

- Inspect intermediate outputs at each stage
- Debug translation quality issues
- Test different configurations
- Understand how each component works

## Pipeline Overview

1. **Setup & Configuration** - Load dependencies and API clients
2. **Input Configuration** - Set video URL and target language
3. **Video Info Retrieval** - Fetch metadata without downloading
4. **Video Download** - Download and optionally trim the video
5. **Audio Extraction** - Extract audio for processing
6. **Audio Separation** - Separate vocals from background music
7. **Speaker Diarization** - Identify different speakers
8. **Voice Sample Extraction** - Extract samples for voice cloning
9. **Transcription** - Convert speech to text
10. **Translation** - Translate text to target language
11. **Speech Synthesis** - Generate translated speech
12. **Audio Mixing** - Mix speech with background
13. **Subtitle Generation** - Create SRT subtitles
14. **Final Merge** - Combine everything into final video

## Cell 1: Setup and Configuration

This cell loads all required dependencies and initializes API clients.

### Dependencies

- **yt-dlp**: Downloads videos from YouTube and 1700+ sites
- **openai**: OpenAI Whisper API for speech-to-text transcription
- **replicate**: Replicate API for:
  - Demucs (audio separation)
  - Speaker diarization
  - Llama (LLM translation)
  - Chatterbox (TTS with voice cloning)
- **deep_translator**: Google Translate fallback for translation
- **httpx**: Async HTTP client for file downloads
- **numpy**: Audio data manipulation

### Environment Variables

Required API keys (loaded from `.env`):
- `OPENAI_API_KEY`: For Whisper transcription
- `REPLICATE_API_TOKEN`: For all Replicate models

Optional configuration:
- `OUTPUT_DIR`: Where to save output files (default: `/tmp/yt-translate`)
- `OXYLABS_PROXY`: Proxy for YouTube downloads (to avoid rate limits)
- `YOUTUBE_COOKIES`: Netscape cookie file for authenticated downloads

In [ ]:
# =============================================================================
# Setup and Configuration
# =============================================================================

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Standard library imports
import asyncio
import os
import re
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Optional

# Third-party imports
import httpx
import numpy as np
import replicate
from replicate.helpers import FileOutput
import yt_dlp
from deep_translator import GoogleTranslator
from openai import OpenAI

# Project imports - language mappings
from languages import (
    SUPPORTED_LANGUAGES,
    GOOGLE_LANG_CODES,
    ISO_639_1_TO_639_2,
    LANG_NAMES,
    get_language_name,
    get_google_code,
    get_iso_639_2_code,
)

# Project imports - configuration constants
from config import (
    GOOGLE_TRANSLATE_CHUNK_SIZE,
    LLM_MAX_SEGMENTS_PER_BATCH,
    LLM_BATCH_OVERLAP,
    LLM_MAX_TOKENS,
    LLM_TEMPERATURE,
    CHATTERBOX_SAMPLE_RATE,
    MIN_VOICE_SAMPLE_DURATION,
    MAX_VOICE_SAMPLE_DURATION,
    SIGNED_URL_EXPIRATION,
    REPLICATE_MODEL_LLAMA,
    REPLICATE_MODEL_CHATTERBOX,
    REPLICATE_MODEL_DEMUCS,
    REPLICATE_MODEL_DIARIZATION,
)

# Project imports - audio utilities
from utils.audio_helpers import read_wav_file, write_wav_file, resample_audio

# =============================================================================
# Helper Functions from cloud_translate.py
# =============================================================================

def parse_timestamp(ts: str) -> float:
    """
    Convert a timestamp string to float seconds.
    
    Handles format like "0:00:00.497812" (H:MM:SS.microseconds)
    or "1:30:45.5" (H:MM:SS.fraction).
    
    Args:
        ts: Timestamp string in format "H:MM:SS.fraction" or "H:MM:SS"
    
    Returns:
        Time in seconds as float (e.g., "1:30:45.5" -> 5445.5)
    """
    parts = ts.split(":")
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = float(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    elif len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    else:
        return float(ts)


def format_timestamp_srt(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format (HH:MM:SS,mmm).
    
    Args:
        seconds: Time in seconds
    
    Returns:
        Timestamp string in SRT format
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


# =============================================================================
# Configuration from Environment
# =============================================================================

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")

# Output directory
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "/tmp/yt-translate"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proxy configuration (optional)
OXYLABS_PROXY = os.getenv("OXYLABS_PROXY")
YOUTUBE_COOKIES = os.getenv("YOUTUBE_COOKIES")

# =============================================================================
# Initialize API Clients
# =============================================================================

# OpenAI client for Whisper API
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# Replicate client is initialized via environment variable automatically
# It reads REPLICATE_API_TOKEN from the environment

# =============================================================================
# yt-dlp Configuration Helper
# =============================================================================

def get_ytdlp_opts(extra_opts: dict = None) -> dict:
    """Get yt-dlp options with optional proxy and cookie support."""
    opts = {'quiet': True, 'no_warnings': True}
    if OXYLABS_PROXY:
        opts['proxy'] = OXYLABS_PROXY
    if YOUTUBE_COOKIES:
        # Write cookies to temp file for yt-dlp
        cookie_file = OUTPUT_DIR / "youtube_cookies.txt"
        cookie_file.write_text(YOUTUBE_COOKIES)
        opts['cookiefile'] = str(cookie_file)
    if extra_opts:
        opts.update(extra_opts)
    return opts


# =============================================================================
# Display Configuration Status
# =============================================================================

print("=" * 60)
print("CONFIGURATION STATUS")
print("=" * 60)
print()
print(f"Output Directory: {OUTPUT_DIR}")
print(f"  - Exists: {OUTPUT_DIR.exists()}")
print()
print("API Keys:")
print(f"  - OPENAI_API_KEY: {'Set (' + OPENAI_API_KEY[:8] + '...)' if OPENAI_API_KEY else 'NOT SET'}")
print(f"  - REPLICATE_API_TOKEN: {'Set (' + REPLICATE_API_TOKEN[:8] + '...)' if REPLICATE_API_TOKEN else 'NOT SET'}")
print()
print("API Clients:")
print(f"  - OpenAI Client: {'Initialized' if openai_client else 'NOT INITIALIZED (missing API key)'}")
print(f"  - Replicate: {'Available' if REPLICATE_API_TOKEN else 'NOT AVAILABLE (missing API token)'}")
print()
print("Optional Configuration:")
print(f"  - Proxy (OXYLABS_PROXY): {'Configured' if OXYLABS_PROXY else 'Not configured'}")
print(f"  - YouTube Cookies: {'Configured' if YOUTUBE_COOKIES else 'Not configured'}")
print()
print("Replicate Models:")
print(f"  - LLM (Llama): {REPLICATE_MODEL_LLAMA}")
print(f"  - TTS (Chatterbox): {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
print(f"  - Audio Separation (Demucs): {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
print(f"  - Speaker Diarization: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
print()
print("Pipeline Constants:")
print(f"  - Chatterbox Sample Rate: {CHATTERBOX_SAMPLE_RATE} Hz")
print(f"  - Min Voice Sample Duration: {MIN_VOICE_SAMPLE_DURATION} seconds")
print(f"  - Max Voice Sample Duration: {MAX_VOICE_SAMPLE_DURATION} seconds")
print(f"  - LLM Max Segments per Batch: {LLM_MAX_SEGMENTS_PER_BATCH}")
print(f"  - LLM Batch Overlap: {LLM_BATCH_OVERLAP}")
print(f"  - LLM Temperature: {LLM_TEMPERATURE}")
print()
print(f"Supported Languages: {len(SUPPORTED_LANGUAGES)}")
print(f"  {', '.join(SUPPORTED_LANGUAGES.values())}")
print()
print("=" * 60)
print("Setup complete! Proceed to the next cell to configure input.")
print("=" * 60)